# Enhance Prompting Startegies
This notebook focuses on enhancing the prompting stategies used for retrieval. At the moment, the model hallucinates same tools which leads to errors in the outputs. 
Goal is now to enhance the prompts such that the model really relies on the given tools from the toolshed and is less liekly to hallucinate tools that are non existent or only exists in other versions.

### Prompts to change
In the current functions, the following prompt needs to be changed: **build_task_prompt**

-> this prompt builds the .ga file based on the retrieved tools from the tool shed metadata file.

-> right now, it just says:
*"Get the tool ids, content ids and versions correct based on the following knowledge base of Galaxy tools: {hits}"*

try adding to **build_task_prompt_enhanced:**
*"NEVER come up with own tool IDs, content IDs or versions on your own. ONLY use the ones provided from the previously given knowledge base."*

In [1]:
# use the evaluation pipeline from eval_notebook.ipynb

from translate import translate_knime_to_galaxy
import re
import os

knime_path = "../Knime_WFs"

# 1. sending to the knime2galaxy pipeline
files = os.listdir(knime_path)
regex_names = []

for f in files:
    name = re.sub(r'^\d{4}_\d{2}_|\.knwf$', '', f)
    regex_names.append(name)
    print(name)

resize_rotate_nested
image_conversion
resize_rotate
segmentation_morph_ope_nested
2D_spot_detection
segmentation_morph_ope
image_conversion_nested


In [2]:
new_folder = "data/output_file/prompt_enhance_test/"
os.makedirs(new_folder, exist_ok=True)
finished_files = os.listdir(new_folder)

for i,file in enumerate(files):
    filename = name = re.sub(r'^\d{4}_\d{2}_|\.knwf$', '', file)
    filename_ga = filename + "_v2.ga"
    if filename_ga in finished_files:
        print(f"File {file} already processed - skipping")
        continue 
    else:
        workflow = translate_knime_to_galaxy(
            knwf_path= f"../Knime_WFs/{file}",
            tools_metadata_path="data/tools_metadata.json",
            translation_table_path="data/translation_table.yml",
            workflow_examples_yml_path="data/workflow_translation_table.yml",
            output_galaxy_workflow_path=os.path.join(new_folder, f"{regex_names[i]}.ga"))

[[{'description': 'Input Image dataset', 'Galaxy': '"0": {\n        "annotation": "",\n        "content_id": null,\n        "errors": null,\n        "id": 0,\n        "input_connections": {},\n        "inputs": [\n            {\n                "description": "",\n                "name": "Input Image Dataset"\n            }\n        ],\n        "label": "Input Image Dataset",\n        "name": "Input dataset",\n        "outputs": [],\n        "position": {\n            "left": 0.0,\n            "top": 0.0\n        },\n        "tool_id": null,\n        "tool_state": "{\\"optional\\": false, \\"tag\\": null}",\n        "tool_version": null,\n        "type": "data_input",\n        "uuid": "312178b3-3fac-49ab-95cf-29e3f44e6015",\n        "when": null,\n        "workflow_outputs": []\n    },\n', 'Python': "from skimage import io\n\ninput_image = io.imread('input_image.tiff')\n", 'KNIME': '<?xml version="1.0" encoding="UTF-8"?>\n<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi

In [5]:
# 2. use worklfow_lint to check for valid .ga file structure 
import subprocess

results = []
outputs = os.listdir(new_folder)
valid_results = 0

for output in outputs:
    if output.startswith("."):
        continue
    result = subprocess.run(
        ["planemo", "workflow_lint", os.path.join(new_folder, output)],
        capture_output=True,
        text=True
    )
    stdout = result.stdout
    stderr = result.stderr
    has_error = "ERROR" in stdout or "ERROR" in stderr

    if has_error:
        print("Lint errors present for workflow", output)
        print(stdout)
    else:
        print("Lint passed for workflow", output)
        valid_results += 1

    results.append(result)
print(f"Valid results: {valid_results} out of 7")

Lint passed for workflow segmentation_morph_ope_nested.ga
Lint errors present for workflow 2D_spot_detection.ga
.. WARNING: Creator identifier "0000-0003-2104-9519" should be a fully qualified URI, for example "https://orcid.org/0000-0002-1825-0097".
.. WARNING: Workflow does not specify a license.
.. WARNING: Workflow step with ID 0 has no annotation.
.. WARNING: Workflow step with ID 0 has no label.
.. WARNING: Workflow step with ID 1 has no annotation.
.. WARNING: Workflow step with ID 1 has no label.
.. WARNING: Workflow step with ID 2 has no annotation.
.. WARNING: Workflow step with ID 2 has no label.
.. WARNING: Workflow step with ID 3 has no annotation.
.. WARNING: Workflow step with ID 3 has no label.
.. WARNING: Workflow step with ID 4 has no annotation.
.. WARNING: Workflow step with ID 4 has no label.
.. WARNING: Workflow step with ID 5 has no annotation.
.. WARNING: Workflow step with ID 5 has no label.
.. WARNING: Workflow step with ID 6 has no annotation.
.. WARNING: Wor

In [6]:
# check whether the error tools are part of the prompting data or whether it is hallucinated by the model
error_tool_1 = "toolshed.g2.bx.psu.edu/repos/jjohnson/query_tabular/filter_tabular/5.0.0"

import json

with open("data/tools_metadata.json") as f:
    data = json.load(f)

def search_values(obj, search_term):
    if isinstance(obj, dict):
        for v in obj.values():
            if search_values(v, search_term):
                return True
    elif isinstance(obj, list):
        for item in obj:
            if search_values(item, search_term):
                return True
    elif isinstance(obj, str):
        if search_term in obj:
            return True
    return False

print(search_values(data, error_tool_1))

True


### the given tool is actually part of the toolshed metadata file, but it still raises an error.
-> need to update the metadata file with all current versions!